# DTU 46500 :  Exercise 6
## Statistical extrapolation for Extreme Winds
#### Fitting extreme distributions, uncertainty in fitting (including sample-length)

In [1]:
####################################
# Special code for Jupyter Notebook
####################################
# Show plots in the notebook
%matplotlib inline

# Automatically reload any packages that changed
%load_ext autoreload
%autoreload 2

# Display multiple arguments
from IPython.core.interactiveshell import InteractiveShell 
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import numpy as np
import datetime
import scipy as sp
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 14})

In [3]:
Data = np.genfromtxt('Hovsore2004to2020_U+dir.csv',delimiter=',')
Dnames = ['Timestamp','Wsp','MeanDir']

In [4]:
# CREATE A DATETIME ARRAY SO WE CAN USE ∆T for removing bad data
T = np.empty(Data[:,0].shape, dtype = datetime.datetime) # Empty numpy array of type datetime
for i in range(len(Data[:,0])):
    T[i] = datetime.datetime.strptime(str(Data[i,0].astype('int64')),'%Y%m%d%H%M')
DeltaT = np.diff(T)

The measurement data is filtered to remove unfeasible data points.\
 Reasonable values are picked for minimum U, maximum U, and maximum acceptable jump in 10-minute mean windspeed from one record to the next:

In [ ]:
LowestWindSpeed = 0.3   # [m/s] anemometer's calm threshold
CrazyWind = 50        # [m/s] 10-minute-mean wind in Denmark has never been this high
maxDeltaU10min = 12.    # [m/s] the 10-minute mean wind should not jump this much from one period to the next

In [ ]:
## check that the 10-minute mena "jump" (increment) limit is reasonable
plt.yscale('log'); plt.hist(np.diff(Data[:,1]),bins=100);

In [ ]:
# CREATE data-selection "masks" (by using Boolean T/F arrays)

WspOK = ((Data[:,1] > LowestWindSpeed)&(Data[:,1] < CrazyWind)) # Boolean (T or F): reject unrealistic wind speeds

DeltaU = np.diff(Data[:,1])
dU10min = np.empty(DeltaU.shape,dtype = 'float')
for i in range(len(DeltaU)):
    dU10min[i] = DeltaU[i]/(DeltaT[i].seconds/600) 
DeltaUOK = (DeltaU > -maxDeltaU10min) & (DeltaU < maxDeltaU10min) 

OK1 = (WspOK[0:-1]==True) & (DeltaUOK == True)
OK2 = (WspOK[1:]==True) & (DeltaUOK == True)
OK1 = np.append(OK1,True)
OK2 = np.insert(OK2,0,True)

DataOK = OK1 & OK2 # Final filter in terms of a Boolean array

In [ ]:
# APPLY FILTERS AND COMPUTE YEAR/MONTH VALUES 

T = T[DataOK]
Timestamp = Data[DataOK,0]
Wsp = Data[DataOK,1]
wdir = Data[DataOK,2]

# Compute year and month
Year = np.floor(Timestamp/1e8).astype('int64')
Month = np.floor(np.mod(Timestamp,1e8)/1e6).astype('int64')

In [ ]:
# plot the filtered data to check it:
plt.figure(figsize=(12, 6))
plt.plot(T,Wsp, label='$U_{filtered}$')
plt.hlines([LowestWindSpeed, CrazyWind], T.min(), T.max(), 'r', label='$V$ filter threshold')
plt.xlabel("$T$ [years]")
plt.ylabel("$V$ [m/s]")
plt.legend()
plt.show();

As can be seen from the above figure the filtered windspeed respects the upper and lower threshold.

In [ ]:
# COMPUTE ANNUAL MAXIMA

Years = np.unique(Year)
nyears = np.max(Years) - np.min(Years)

Umax = np.empty(nyears,dtype = 'float')
## since data starts in October, make each "year" start then
for i in range(nyears):
    CurrentPeriod = ((Year==Years[i]) & (Month >= 10)) | ((Year == (Years[i]+1)) & (Month < 10))
    Umax[i] = max(Wsp[CurrentPeriod])

In [ ]:
np.sort(Umax)

## GUMBEL FIT - Method of Probability-weighted moments (PWM)
Use PWM to compute $\{\alpha,\beta\}$ via $\{b_0,b_1\}$ using the 16 years of data provided. 

Then use $\{\alpha,\beta\}$ to estimate $V_{50}$. 

You'll do this a number of times; 
- first, based on the whole dataset, 
- then again, using 2-year non-overlapping segments you'll find 7 "two-year" values of $\{\alpha,\beta,V_{50}\}$. 

The framework for code below helps you start out.

You'll also need to **answer quantitatively: "how well does this follow a Gumbel distribution!?" for both cases.**  
Try to quantify the variation of using only 2-year periods; how does this compare to theoretical expectation? 

To solve, 3 functions are defined:
- `gumbel_par()`\
A function to calculate the Gumbel parameters $\alpha$ and $\beta$ using the Probability Weighted Momemnts (PWM) method. (Slides 6ab, page 20)
- `gumbel_extreme_wind()`\
A function to calculate the $T$-year extreme wind with a base period of $T_0 = 1$yr using the PWM method (Slides 6ab, page 21)
- `gumbel_mc_var()`\
Ott's estimate of variance (like uncertainty squared) around extreme $U(T)$ due to Gumbel-fit (Slides, page 21)

In [ ]:
gamma = 0.577216 # Euler-Mascheroni constant

#### PWM: calculate b0 and b1, find Gumbel parameters (alpha, beta) from them
def gumbel_par(Umax):
    UmaxSorted = np.sort(Umax)
    N = len(Umax)
    b0PWM = 1 / N * np.sum(UmaxSorted)
    indexes = np.arange(len(UmaxSorted)) + 1
    b1PWM = 1 / N * np.sum((indexes - 1) / (N - 1) * UmaxSorted)
    alphaFit = (2 * b1PWM - b0PWM) / np.log(2)
    betaFit = b0PWM - gamma * alphaFit
    return alphaFit, betaFit, b0PWM, b1PWM

#### for T-year return period, using T0=1year, given {alpha,beta} find U(T)
def gumbel_extreme_wind(T, alpha, beta):
    Vxx = beta + (alpha * np.log(T/1))
    return (Vxx)

#### find the expected variance in the PWM Gumbel-fit using Ott's formula
def gumbel_mc_var(T, alphaPWM, N, T0=1):
    qt = (np.log(T/T0) - gamma) / np.log(2)
    var = alphaPWM**2*np.pi**2 / (6*N) * (1 + 0.584*qt + 0.234*qt**2 / (1-0.823/N))
    return var

In [ ]:
alphaPWM, betaPWM, b0PWM, b1PWM = gumbel_par(Umax)
[alphaPWM,betaPWM, b0PWM, b1PWM]

In [ ]:
V50PWM = gumbel_extreme_wind(50, alphaPWM, betaPWM)
print(f"PWM results: V50 = {V50PWM.round(2)} [m/s], beta = {betaPWM.round(2)} [m/s], alpha = {alphaPWM.round(2)} [m/s]");

The fitted Gumbel PDF is shown below. 

In [ ]:
from scipy import stats
UmaxSorted = np.sort(Umax)
gumbel = stats.gumbel_r(loc=betaPWM, scale=alphaPWM)
plt.plot(np.linspace(np.min(Umax), np.max(Umax), 100), gumbel.pdf(np.linspace(np.min(Umax), np.max(Umax), 100)), "k")
plt.xlabel("$U_T$ [m/s]")
plt.ylabel("PDF")
plt.show();

Instead of using the full dataset $V_{50}$ now estimate using chunks of 2 year, many times.

**How good is a Gumbel-fit of annual maxima, using only 2 years?  What is the uncertainty?**

In [ ]:
Umax_2year = Umax.reshape(8, 2)
Umax_2year = Umax_2year[:-1, :]
V50PWM_2years = []
V50PWM_var_2years = []
alphaPWM_2years = []

for i, U_max in enumerate(Umax_2year):
    alphaPWM_, betaPWM_, b02y_, b12y = gumbel_par(U_max)
    V50PWM_2years.append(gumbel_extreme_wind(50, alphaPWM_, betaPWM_))

print(f"V_50_2y = {np.round(V50PWM_2years, 2)} m/s", "\n", f"std_V50_2y = {np.std(V50PWM_2years).round(2)} m/s")

Going back to the Gumbel-fit based on the full dataset (`gumbel` found above using `gumbel_r` from `scipy.stats`), the CDF is plotted below.

In [ ]:
plt.plot(np.linspace(np.min(Umax), np.max(Umax)+9, 50), gumbel.cdf(np.linspace(np.min(Umax), np.max(Umax)+9, 50)), "k")
FUmax = np.arange(1, N + 1 ,1)/(N + 1)
plt.scatter(UmaxSorted, FUmax)
plt.xlabel("$U_T$ [m/s]")
plt.ylabel("CDF")
plt.show();

The fitted gumbel distribution CDF is plotted along with the data-based empirical CDF by data ranking. As can be seen from the plot, some discrepancy between the two is observed. 

Given assumed independence, the expected occurence intervals for the measured maxima $(N_{years}^{i/N_{years}})$ is plotted against the fitted Gumbel distribution on a semi-log plot.\
To show how well the measured data fits the Gumbel expression $2 \cdot \sigma$ is shown with the fit i.e. the 95% confidence intreval.
Both the Abild and Ott variance expresions are shown.

In [ ]:
ToT0ranked = ((nyears)/1)**(np.arange(1, nyears+1)/nyears)

In [ ]:
N = nyears
plt.figure(figsize=(16,8))
years = np.linspace(0.1, 50, 100)
plt.semilogx(years, betaPWM + alphaPWM*np.log(years), "k", label="Gumbel fit")
n_stds = 2
stds_mc = np.sqrt(gumbel_mc_var(years, alphaPWM, N))
plt.semilogx(years, betaPWM + alphaPWM*np.log(years)+stds_mc*n_stds, 'r--', alpha=0.75, label=f"$\pm{n_stds}\sigma_{{mc}}$")
plt.semilogx(years, betaPWM + alphaPWM*np.log(years)-stds_mc*n_stds,  'r--', alpha=0.75)
plt.semilogx(ToT0ranked, UmaxSorted, "x", label="$V_{max, meas}$")
plt.grid(True)
plt.xlabel("$T/T_0$ [number of years]")
plt.ylabel("$U_{T}$ [m/s]")
plt.legend()
plt.show();

To assess the goodness of fit of the derived Gumbel distributions based on the method of PWM, the Kolmogorov-Smirnov test can be applied. The null-hypothesis being tested is that the 16-years of measurement data belongs to the Gumbel distribution estimated based on PWM. Using `scipy.stats.kstest`, the p-value significance level is 0.05 for this test.

In [ ]:
ksPWM = stats.kstest(UmaxSorted,'gumbel_r', [betaPWM, alphaPWM])
print(f'Probability Weighted Moments: Kolmogorov-Smirnov test p-value: {np.round(ksPWM[1], 2)}')

does this show that the data follows a Gumbel-distribution?

Try a different fit here, to see what $U_{T=50y}$ you can get. \
One example: considering Gumbel CDF, the extreme event return period ($T$) for a given $U_T$ is\
 $\quad$ $T = \frac{T_0}{1-F(U_T)}$  since $F(U_T) = 1-\frac{T_0}{T}$; using this could extrapolate higher values...

### GUMBEL FIT - V50 from Maximum-Likelihood Method
The Max-Likelihood Method (MLM) is chosen as an alternative to PWM.\
Below, the Gumbel parameters are fitted using MLM (Slides 1c - page 39)


In [ ]:
from scipy import optimize
gumbel_likelihood = lambda theta: -np.sum(np.log(stats.gumbel_r.pdf(Umax,loc=theta[0], scale=theta[1]))) # Definition of the negative log-likelihood function
gumbel_opt = optimize.minimize(gumbel_likelihood,[10,5]) # Minimization to find the optimal distribution parameters

betaMLM, alphaMLM = gumbel_opt.x[0], gumbel_opt.x[1]
V50MLM_eng = betaMLM + alphaMLM * np.log(50)

ksMLM = stats.kstest(UmaxSorted,'gumbel_r', [betaMLM, alphaMLM])

The two fits are plotted against each other for comparison and $V_{50}$, $\alpha$ and $\beta$ are printed. 

In [ ]:
## Plot PWM vs max-likelihood
plt.figure()
U_plt = np.linspace(np.min(Umax), np.max(Umax), 100)
label_string_pwm = f"{betaPWM.round(2)}$\\cdot\\log(T_{{es}})+ {alphaPWM.round(2)}$"
plt.plot(U_plt, stats.gumbel_r.pdf(U_plt,loc=betaPWM, scale=alphaPWM),'-b', label=f'PWM')
plt.plot(U_plt, stats.gumbel_r.pdf(U_plt,loc=betaMLM, scale=alphaMLM),'--r', label='Max-likelihood')
plt.legend()
plt.xlabel("$V_{max}$ m/s")
plt.ylabel("PDF")
plt.grid()
plt.show();
print(f"PWM: V50 = {V50PWM.round(2)} [m/s], beta = {betaPWM.round(2)} [m/s], alpha = {alphaPWM.round(2)} [m/s]");
print(f"MLM: V50 = {V50MLM_eng.round(2)} [m/s], beta = {betaMLM.round(2)} [m/s], alpha = {alphaMLM.round(2)} [m/s]");
print(f'MLM: Kolmogorov-Smirnov test p-value: {np.round(ksMLM[1], 2)}');

The Kolmogorov-Smirnov test for the MLM fit shows a p-value very close to the corresponding test to for PWM fit i.e. the null hypothesis cannot be rejected for either fit.  

# Ex.6(b): Uncertainty in $V_{50}$ estimates
Using the $\alpha$-parameter you found above in [1a] using _the whole 16-year timeseries_, 
- do bootstrap sampling with many 2-year segments to get a distribution of $f_{2y}(\alpha$ and $f_{2y}(V_{50})$; 
     - what is the uncertainty in using 2-year segments, i.e., the standard deviation $\sigma_{V50_\textrm{fit 2yr}}$
- repeat this for other segment lengths (3 years, 4y...10y), to get (fit) a relation for $\sigma^2_{V50}(T_\textrm{segment})$
- How does your relation for $\sigma^2_{V50}(T_\textrm{segment})$ compare to the Ott(2011) expression for $\sigma^2_{U_T}$? 
    - ignoring $\beta$, how much variation in $\alpha$ (e.g. $\sigma_\alpha/\langle\alpha\rangle$) does your $\sigma^2_{V50}(T_\textrm{segment})$ correspond to? 

Setting up a Gumbel distrbution based on the fit from 1a)
1000 random samples of annual wind maxima are simulated from this distrbution.\
The samples are shown as a histogram with 50 bins, illustrating the Gumbel shape.

In [ ]:
gumbel_bootstrap = stats.gumbel_r(scale=alphaPWM, loc=betaPWM)
mc_samples = gumbel_bootstrap.rvs(size=1000)
plt.figure()
plt.hist(mc_samples, 50)
plt.xlabel("U")
plt.xlabel("$U_{max}$ [m/s]")
plt.ylabel("Count [-]")
plt.grid(True)
plt.show();

To perform bootstrap sampling a function is defined taking the original sample, size of bootstrap sample and amount of bootstrap samples.

In [ ]:
def BootstrapSampler(org_sample, BootstrapSize, Nbootstrap=99999):
    """Make bootstrap samples based on original input and bootstrap parameters
    """
    Bsample = np.random.randint(low = 0, high = len(mc_samples), size = (Nbootstrap,BootstrapSize))
    BootstrapSample = np.zeros((Nbootstrap,BootstrapSize))
    for i in range(Nbootstrap):
        for j in range(BootstrapSize):
            BootstrapSample[i,j] = org_sample[Bsample[i,j]]
    return BootstrapSample

Using the defined bootstrap function samples are drawn from the simulated annual wind maxima data.
$\alpha$, $\beta$ and resulting $V_{50}$ is found using the previously defined PWM routines. Here the bootstrap sample size is low with 2 simulated maxima pr. sample.

In [ ]:
np.random.seed(1) # Set random seed to get same result when sampling
BootstrapSize = 2
BootstrapSample = BootstrapSampler(mc_samples, BootstrapSize=BootstrapSize)

alphas = [gumbel_par(sample)[0] for sample in (BootstrapSample)]
betas = [gumbel_par(sample)[1] for sample in (BootstrapSample)]
V50s = [gumbel_extreme_wind(50, alpha, beta) for alpha, beta in zip(alphas, betas)]

In [ ]:
plt.figure()
plt.hist(V50s, 100)
plt.xlabel("$V_{50}$ [m/s]"); plt.ylabel("Count [-]"); plt.grid(True)
plt.show();

plt.figure()
plt.hist(alphas, 100); plt.yscale('log');
plt.xlabel("$\\alpha$ [m/s]"); plt.ylabel("Count [-]"); plt.grid(True)
plt.show();

In [ ]:
V50s_std = np.std(V50s)
print(f"V50_std = {V50s_std.round(2)} m/s")

The Bootstrap-sampling technique is repeated for larger bootstram sample sizes, from $T_{es}=3$ to 10 years.

In [ ]:
T_es = np.arange(2, 10+1)
V50s_stds = []
V50s_vars = []
mean_alphas = []
ott_stds = []
for T in T_es:
    BootstrapSample_ = BootstrapSampler(mc_samples, BootstrapSize=T)
    alphas_ = [gumbel_par(sample)[0] for sample in (BootstrapSample_)]
    betas_ = [gumbel_par(sample)[1] for sample in (BootstrapSample_)]
    V50s_ = [gumbel_extreme_wind(50, alpha, beta) for alpha, beta in zip(alphas_, betas_)]
    V50s_stds.append(np.std(V50s_))
    V50s_vars.append(np.var(V50s_))
    mean_alphas.append(np.mean(alphas_))
    alpha_quantiles = np.quantile(alphas_, [.25,.5,.75])
    ott_stds.append(np.sqrt([gumbel_mc_var(50, alpha_, T) for alpha_ in alpha_quantiles]))
    # plt.hist(alphas_, 100);
    # plt.show();
ott_stds = np.array(ott_stds)

A function is fitted for the relation between the observed $\sigma_{V50}$ and the bootstrap sample size ($T_{es}$).
The functions is assumed to be of the form $\sigma_{V50} = a + b T_{es}^{-1/2}$

In [ ]:
from scipy.optimize import curve_fit
inv_par = curve_fit(lambda t,a,b: a+b*1/np.sqrt(t),  T_es,  V50s_stds)
inv_fit = lambda t: inv_par[0][0]+inv_par[0][1]*1/np.sqrt(t)
T_es_ls = np.linspace(T_es[0]-1, T_es[-1]+2, 100)

In the plot below the fitted function is plotted against the observed bootstrap standard deviations of $V_{50}$ ($\sigma_{Bootstrap}$), the mean $\sigma_{Ott}$ and the $\sigma_{Ott}$ Q25 and Q75 quantiles.

In [ ]:
plt.figure(figsize=(16,12))
plt.fill_between(T_es, ott_stds[:, 0], ott_stds[:, 2], color="grey", alpha=.5, label="$\\sigma_{Ott}: \\alpha_{Q25} \\ to \\  \\alpha_{Q75}$")
label_string_inv = "fitted $\\propto T^{-1/2}$"
plt.plot(T_es_ls, inv_fit(T_es_ls), "--g", label=label_string_inv)
mc_stds = np.sqrt([gumbel_mc_var(50, mean_alpha, T_) for mean_alpha, T_ in zip(mean_alphas, T_es)])
plt.scatter(T_es, V50s_stds, label="$\\sigma_{Bootstrap}$", marker="o", s= 75)
plt.scatter(T_es, mc_stds, marker="x", s=75, c="red", label="Mean $\\sigma_{Ott}$")
# plt.plot(T_es_ls, poly2_fit(T_es_ls))
plt.xlabel("$T_{es}$ [years]")
plt.ylabel("$\\sigma_{V50}$ [m/s]")
plt.grid(True)
plt.legend()
plt.show();